In [1]:
import json
from pathlib import Path

#now lets get the actual requested documents. Just the normal one
REQS_DOCS_2 = '../requested_documents.json'
with open(REQS_DOCS_2, 'r') as f:
    requested_documents_2 = json.load(f)

REQS_DOCS_firstIter = '../prev_requesteddocs/requested_documents_FirstIter.json'
with open(REQS_DOCS_firstIter, 'r') as f:
    requested_documents_first_iter = json.load(f)

consolidated_requested_documents = {}

for k,v in requested_documents_first_iter.items():
    consolidated_requested_documents[k] = v

for k,v in requested_documents_2.items():
    if k not in consolidated_requested_documents:
        consolidated_requested_documents[k] = v
    else:
        for files in v:
            if files not in consolidated_requested_documents[k]:
                consolidated_requested_documents[k].append(files)

total_files = 0
for k,v in consolidated_requested_documents.items():
    total_files += len(v)

with open("consolidated_requested_documents.json", "w") as f:
    json.dump(consolidated_requested_documents, f, indent=4)

In [2]:

AUTOMATION_AUDIT = Path("automation_audit.json")

with open(AUTOMATION_AUDIT, 'r') as f:
    automation_audit = json.load(f)


total_aut_files = 0
total_files = 0

for k,v in automation_audit.items():
    total_aut_files += len(v)

for k,v in consolidated_requested_documents.items():
    total_files += len(v)

print("Total files: ", total_files)
print("Total aut files: ", total_aut_files)
print("diff: ", total_files - total_aut_files)

Total files:  8015
Total aut files:  7952
diff:  63


In [3]:
total = 0
missing_people = set()
for k, v in consolidated_requested_documents.items():
    if k in automation_audit:
        if len(v) > len(automation_audit[k]):
            missing_people.add(k)
            total += len(v) - len(automation_audit[k])

print(len(missing_people))
print(total)

93
158


In [19]:
new_reqs_docs = {}
total_files_needed = 0
for k,v in consolidated_requested_documents.items():
    if k not in automation_audit:
        continue
    else:
        if len(v) > len(automation_audit[k]):
            new_reqs_docs[k] = v
            total_files_needed += len(v) - len(automation_audit[k])

with open("new_reqs_docs.json", "w") as f:
    json.dump(new_reqs_docs, f, indent=4)

print(total_files_needed)

187


Making the table

In [4]:
import pandas as pd

#a table with the following columns:
# Name, files recived, file names, remaining files
#export to excel

cols = ["Name", "files recived", "file names", "remaining files"]
df = pd.DataFrame(columns=cols)
row_list = []
for k,v in automation_audit.items():
    if k in consolidated_requested_documents:
        row_list.append({
            "Name": k,
            "#files recived": len(v),
            "file names": v,
            "remaining files": max(0, len(consolidated_requested_documents[k]) - len(v))
        })

df = pd.DataFrame(row_list)
df.to_excel("table.xlsx", index=False)